# Tools with MCP ⏰

The Model Context Protocol (MCP) provides a standardized way to connect AI agents to external tools and data sources. Let's connect to an MCP server using `@langchain/mcp-providers`.

In [1]:
import { MultiServerMCPClient } from "@langchain/mcp-adapters";

// Connect to the mcp-time server for timezone-aware operations
// This Go-based server provides tools for current time, relative time parsing,
// timezone conversion, duration arithmetic, and time comparison
const mcpClient = new MultiServerMCPClient({
    mcpServers: {
        time: {
            transport: "stdio",
            command: "npx",
            args: ["-y", "@theo.foobar/mcp-time"],
        }
    },
    useStandardContentBlocks: true,
});

// Load tools from the MCP server
const mcpTools = await mcpClient.getTools();
console.log(`Loaded ${mcpTools.length} MCP tools:`, mcpTools.map(t => t.name));

Loaded 5 MCP tools: [
  "add_time",
  "compare_time",
  "convert_timezone",
  "current_time",
  "relative_time"
]


Create an agent with the MCP-provided time tools.

In [2]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";

const agentWithMCP = createAgent({
    model: "anthropic:claude-sonnet-4-5-20250929",
    tools: mcpTools,
    systemPrompt: "You are a helpful assistant"
});


Ask about the current time in San Francisco.

In [3]:
const result = await agentWithMCP.invoke({
    messages: "What's the current time in San Francisco right now?"
})

for (const message of result.messages) {
    displayMessage(message)
}


┌────────────────────────────────────────────────────────────┐


│ 👤 HUMAN MESSAGE                                           │


└────────────────────────────────────────────────────────────┘


What's the current time in San Francisco right now?



┌────────────────────────────────────────────────────────────┐


...

2026-04-04T09:07:29-07:00



┌────────────────────────────────────────────────────────────┐


│ 🤖 AI MESSAGE                                              │


└────────────────────────────────────────────────────────────┘


The current time in San Francisco is **9:07 AM** (Pacific Daylight Time) on Saturday, April 4th, 2026.
